This tutorial shows how to evaluate an MMContext Sentence Transformers model on one dataset. It assumes you created a huggingface dataset, which contains the cell representations (either cell ids for numerical embeddings or cell sentences for text_only usage). Such datasets can be created with a pipeline available through the https://github.com/mengerj/adata_hf_datasets repo. If you instead want to start from an adata object, see the tutorial pretrained_inference.ipynb

Figure 1D in the publication was created with this notebook

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
from datasets import load_dataset

repo_name = "jo-mengr"
dataset_name = "hiha_100k"
split_name = "test"
label_key = "AIFI_L2"  # "AIFI_L2"

In [4]:
dataset = load_dataset(f"{repo_name}/{dataset_name}")
test_dataset = dataset[split_name]

In [5]:
from sentence_transformers import SentenceTransformer

# model_name = "jo-mengr/mmcontext-pubmedbert-gs10k"
model_name = "../outputs/multi_gs10k/final"
model = SentenceTransformer(model_name, trust_remote_code=True)
data_type = "gs10k"
layer_key = f"X_{data_type}"
text_only = False  # set True for text-based models (uses cell_sentence_2)

In [6]:
import os
from pathlib import Path

from mmcontext.embed import prepare_inference

# Cache + store locations are derived from the knobs set at the top of the
# notebook — nothing to edit here:
#   * adata cache depends only on the dataset (same raw data across models/keys)
#   * the vector store is keyed by model + dataset + embedding key, so switching
#     the model or layer_key never silently reuses a stale store.
model_slug = model_name.strip("/").replace("/", "_")
adata_cache_dir = Path("../data/test_adata") / dataset_name
store_path = Path("../outputs/inference_stores") / model_slug / f"{dataset_name}__{layer_key}.mmap"

# One call replaces: load_test_adata_from_hf_dataset + subset_dataset_by_chunk
# + prepare_vector_store + set_vector_store (bimodal) / truncation (text).
# The cell sentence is chosen internally from the modality
# (cell_sentence_1 for bimodal omics, cell_sentence_2 for text).
modality = "text" if text_only else "bimodal"
bundle = prepare_inference(
    model,
    test_dataset,
    modality=modality,
    obsm_key=layer_key,
    cache_dir=str(adata_cache_dir),
    store_path=str(store_path),
    zenodo_token=os.getenv("ZENODO_TOKEN"),
    truncate=text_only,
    truncate_kwargs={"max_length": 64, "filter_strings": ["RPS", "RPL", "MT"]},
)
adata = bundle.adata
dataset_ready = bundle.dataset

Processing:   0%|          | 0/1 [00:00<?, ?file/s]

Processing adata files:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
dataset_to_use = dataset_ready  # [split_name]

In [8]:
text_encoder_name = model[0].model_name_or_path
text_encoder = SentenceTransformer(text_encoder_name)

In [9]:
dataset_to_use[0]

{'anchor': 'omics:cfe6b50ea2e611eb91855e3d154f7c3f',
 'sample_idx': 'cfe6b50ea2e611eb91855e3d154f7c3f',
 'adata_link': 'https://zenodo.org/api/records/17715306/draft/files/all_chunk_0.zarr.zip/content'}

In [10]:
omics_embeddings = model.encode(dataset_to_use["anchor"])
adata.obsm["mmcontext_emb"] = omics_embeddings

In [11]:
n_colours = len(adata.obs["AIFI_L1"].unique())

In [12]:
import seaborn as sns

auto_colors = sns.color_palette("tab10", n_colours)

In [13]:
auto_colors

[(0.12156862745098039, 0.4666666666666667, 0.7058823529411765),
 (1.0, 0.4980392156862745, 0.054901960784313725),
 (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),
 (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),
 (0.5803921568627451, 0.403921568627451, 0.7411764705882353),
 (0.5490196078431373, 0.33725490196078434, 0.29411764705882354),
 (0.8901960784313725, 0.4666666666666667, 0.7607843137254902),
 (0.4980392156862745, 0.4980392156862745, 0.4980392156862745),
 (0.7372549019607844, 0.7411764705882353, 0.13333333333333333)]

In [14]:
label_colors = {
    "T cell": auto_colors[7],
    "B cell": auto_colors[0],
    "NK cell": auto_colors[1],
    "Monocyte": auto_colors[2],
    "DC": auto_colors[3],
    "Platelet": auto_colors[4],
    "Progenitor cell": auto_colors[5],
    "ILC": auto_colors[6],
    "Erythrocyte": auto_colors[8],
}

In [15]:
from mmcontext.eval import get

EvClass = get("LabelSimilarity")
ev = EvClass(
    auto_filter_labels=False,
    umap_n_neighbors=10,
    umap_min_dist=0.4,
    similarity="cosine",
    logit_scale=1,
    score_norm_method=None,
    label_colors=None,
    annotation_fontsize=16,
    font_family="Arial",
)

In [16]:
# precompute umap coordinates to reuse on subset
full_omics_embeddings = adata.obsm["mmcontext_emb"]

In [ ]:
# full_cell_umap = ev._compute_umap(full_omics_embeddings)
# add umap coordinates to adata
# adata.obsm["cell_umap"] = full_cell_umap

In [17]:
full_query_labels = adata.obs[label_key].unique().tolist()
full_label_embeddings = model.encode(full_query_labels)
full_true_labels = adata.obs[label_key]

In [18]:
from pathlib import Path

result = ev.compute(
    omics_embeddings=full_omics_embeddings,
    label_embeddings=full_label_embeddings,
    query_labels=full_query_labels,
    true_labels=full_true_labels,
    label_key=label_key,
    out_dir=Path(f"LabelSimilarity/{model_name}/{dataset_name}"),  # Pass output directory for caching
)

/Users/mengerj/repos/mmcontext/src/mmcontext/eval/label_similarity.py:76: RuntimeWarning: divide by zero encountered in matmul
  return (emb1_norm @ emb2_norm.T).astype(np.float32)
/Users/mengerj/repos/mmcontext/src/mmcontext/eval/label_similarity.py:76: RuntimeWarning: overflow encountered in matmul
  return (emb1_norm @ emb2_norm.T).astype(np.float32)
/Users/mengerj/repos/mmcontext/src/mmcontext/eval/label_similarity.py:76: RuntimeWarning: invalid value encountered in matmul
  return (emb1_norm @ emb2_norm.T).astype(np.float32)


In [19]:
result

ASDC/auc: 0.4956
ASDC/accuracy: 0.0000
CD14 monocyte/auc: 0.9785
CD14 monocyte/accuracy: 0.9417
CD16 monocyte/auc: 0.9181
CD16 monocyte/accuracy: 0.0009
CD56bright NK cell/auc: 0.9277
CD56bright NK cell/accuracy: 0.6163
CD56dim NK cell/auc: 0.9499
CD56dim NK cell/accuracy: 0.0829
CD8aa/auc: 0.6865
CD8aa/accuracy: 0.0000
DN T cell/auc: 0.7996
DN T cell/accuracy: 0.0089
Effector B cell/auc: 0.9421
Effector B cell/accuracy: 0.0516
Erythrocyte/auc: 0.8086
Erythrocyte/accuracy: 0.6000
ILC/auc: 0.4408
ILC/accuracy: 0.0000
Intermediate monocyte/auc: 0.9244
Intermediate monocyte/accuracy: 0.0549
MAIT/auc: 0.7842
MAIT/accuracy: 0.0000
Memory B cell/auc: 0.9552
Memory B cell/accuracy: 0.0521
Memory CD4 T cell/auc: 0.7963
Memory CD4 T cell/accuracy: 0.7317
Memory CD8 T cell/auc: 0.8231
Memory CD8 T cell/accuracy: 0.1451
Naive B cell/auc: 0.9625
Naive B cell/accuracy: 0.3367
Naive CD4 T cell/auc: 0.7379
Naive CD4 T cell/accuracy: 0.4369
Naive CD8 T cell/auc: 0.6935
Naive CD8 T cell/accuracy: 0.000

In [ ]:
ev.plot(
    omics_embeddings=full_omics_embeddings,
    # cell_umap=full_cell_umap,
    out_dir=Path(f"LabelSimilarity/{model_name}/{dataset_name}/{label_key}_combined"),
    label_embeddings=full_label_embeddings,
    query_labels=full_query_labels,
    true_labels=full_true_labels,
    label_key=label_key,  # column name (e.g. "celltype")
    save_format="svg",
    figsize=(2.5, 2.5),
    dpi=300,
    font_size=12,
    font_style="normal",
    font_weight="normal",
    legend_fontsize=54,
    axis_label_size=20,
    axis_tick_size=12,
    point_size=0.25,
    legend_layout="vertical",
    legend_point_size=16,
    umap_method="combined",
    label_min_distance=0.2,
    label_spring_strength=0.5,
    label_repulsion_strength=1,
)

In [ ]:
# Option to subset adata based on one or more label values (e.g., "Monocyte" and "DC")
subset_label_values = ["T cell"]  # Change this list to your desired label values
subset_label_key = "AIFI_L1"
annotation_label_key = "AIFI_L2"
# Subset the AnnData object for any of the specified label values
adata_subset = adata[adata.obs[subset_label_key].isin(subset_label_values)].copy()
subset_labels = adata_subset.obs[annotation_label_key].values.unique()
label_embeddings_subset = model.encode(subset_labels)
# Create a new LabelSimilarity evaluator instance
# ev_subset = EvClass(auto_filter_labels=False, umap_n_neighbors=15, umap_min_dist=0.5)
subset_label_string = "_".join(subset_label_values)
subset_omics_embeddings = adata_subset.obsm["mmcontext_emb"]
# subset_umap_coords = adata_subset.obsm["cell_umap"]
# ev_subset.eb_lfdr_q = 0.01
ev = EvClass(
    auto_filter_labels=False,
    umap_n_neighbors=10,
    umap_min_dist=0.4,
    similarity="cosine",
    logit_scale=1,
    score_norm_method=None,
    font_family="Arial",
    annotation_fontsize=18,
)
# Compute metrics on the subsetted data
result_subset = ev.compute(
    omics_embeddings=subset_omics_embeddings,
    label_embeddings=label_embeddings_subset,
    query_labels=subset_labels,
    true_labels=adata_subset.obs[annotation_label_key],
    label_key=annotation_label_key,
    out_dir=Path(
        f"LabelSimilarity/{model_name}/{dataset_name}/{annotation_label_key}_subset_{subset_label_string}/results"
    ),
)

# Plot results for the subset
ev.plot(
    omics_embeddings=subset_omics_embeddings,
    #    cell_umap=subset_umap_coords,
    out_dir=Path(
        f"LabelSimilarity/{model_name}/{dataset_name}/{annotation_label_key}_subset_{subset_label_string}_combined"
    ),
    label_embeddings=label_embeddings_subset,
    query_labels=subset_labels,
    true_labels=adata_subset.obs[annotation_label_key],
    label_key=annotation_label_key,
    save_format="svg",
    figsize=(2.5, 2.5),
    dpi=300,
    font_size=12,
    axis_tick_size=12,
    font_style="normal",
    font_weight="normal",
    axis_label_size=20,
    point_size=0.25,
    legend_layout="vertical",
    legend_point_size=20,
    umap_method="combined",
    label_min_distance=0.15,
    label_spring_strength=0.35,
    label_repulsion_strength=1.4,
)

In [ ]:
result_subset

In [ ]:
# Visualise the embeddings
from mmcontext.pl import plot_umap
from mmcontext.utils import consolidate_low_frequency_categories

current_key = label_key
adata_cut = consolidate_low_frequency_categories(adata, [current_key], threshold=50, remove=True)
emb_key = "mmcontext_emb"
plot_umap(
    adata,
    color_key=label_key,
    embedding_key=emb_key,
    save_format="svg",
    save_dir=f"figs/{model_name}/{dataset_name}",
    save_plot=False,
    title="",
)

In [ ]:
# Visualise the embeddings
from mmcontext.pl import plot_umap
from mmcontext.utils import consolidate_low_frequency_categories

current_key = label_key
adata_cut = consolidate_low_frequency_categories(adata, [current_key], threshold=1, remove=False)
emb_key = layer_key
plot_umap(
    adata_cut,
    color_key=label_key,
    embedding_key=emb_key,
    save_format="svg",
    nametag="",
    save_dir=f"figs/{model_name}/{dataset_name}",
    save_plot=False,
    title="",
)

In [ ]:
from mmcontext.eval.query_annotate import OmicsQueryAnnotator

annotator = OmicsQueryAnnotator(model)
annotator.annotate_omics_data(adata, full_query_labels, emb_key="mmcontext_emb")

In [ ]:
# get accuracy of best label vs true label
from sklearn.metrics import accuracy_score

accuracy_score(adata.obs["best_label"], adata.obs[label_key])

In [ ]:
queries_csv = "../../data/queries/additional_combined.csv"
if dataset_name == "hiha_100k" and os.path.exists(queries_csv):
    df = pd.read_csv(queries_csv)
    labels = df["Cell Type"]
    Definition = df["Definition"]
    from mmcontext.eval.query_annotate import OmicsQueryAnnotator
    from mmcontext.pl.plotting import plot_query_scores_with_labels_umap

    annotator = OmicsQueryAnnotator(model)
    annotator.query_with_text(adata, Definition, emb_key="mmcontext_emb")
    # Call the plotting function
    plot_query_scores_with_labels_umap(
        adata=adata,
        queries=Definition,
        labels=labels,
        label_key="AIFI_L2",
        save_dir=f"figs/{model_name}/{dataset_name}/umap_with_labels",
        nametag="",
        figsize=(4, 4),
        point_size=2,
        dpi=300,  # Lower DPI for faster generation
        axis_label_size=18,
        axis_tick_size=18,
    )